# Creating inventories based on external data

This notebook shows how we can update the land use factors for palm oil production in Malaysia.

## Case study: Land use of palm oil production in Malaysia

We are building on a project with `ecoinvent-3.10.1-cutoff` already imported. We notice that [ecoinvent claims](https://ecoquery.ecoinvent.org/3.10.1/cutoff/dataset/441/documentation) that 1 kilogram of palm fruit bunch production in Malaysia takes 0.4 square meter-years:

![image](img/ecoinvent-land.png)

Palm fruit plantations take several years to mature, and occupy the land continuously, so we can convert this number to 0.4 square meters per kilogram.

According to [Our world in data](https://ourworldindata.org/grapher/palm-oil-yields?mapSelect=~MYS), Malaysian yield has been pretty consistent for the last decade, and is around 18.5 tonnes per hectare. Let's put that in comparable units:

$1 \div ( 18.5 \frac{tonnes}{hectare} \cdot 1000 \frac{kilogram}{tonne} \div 10000 \frac{square-meter}{hectare} ) = 0.54 \frac{square-meter}{kilogram}$

As land use and land transformation are quite important for palm oil, and these two numbers are already rather difference, let's look in more detail. I found an [official government publication](https://www.kpk.gov.my/kpk/images/mpi_statistik/2023_statistik_on_commodity/Sawit_2023.pdf) from 2023 which lists both planted area and oil production per state from 2018 - 2022. There wasn't much different in time, so we will take 2022 for now, but you could easily modify the below code to generate datasets per year.

We start by loading our existing project:

In [ ]:
import pandas as pd
import geopandas
import numpy as np

import bw2regional as bwr
import bw2data as bd
import bw2calc as bc
import bw2io as bi

In [ ]:
PROJECT_NAME = "Palm-oil"

In [ ]:
bd.projects.set_current(PROJECT_NAME)

Convenience function to reset case study work

In [ ]:
def reset_case_study():
    if 'Malaysian palm land use' not in bd.databases:
        return
    
    del bd.databases['Malaysian palm land use']
    palm_ecoinvent = bd.get_node(
        database="ecoinvent-3.10.1-cutoff",
        name='palm fruit bunch production',
        location="MY"
    )
    for exc in palm_ecoinvent.technosphere():
        try:
            str(exc.input)
        except:
            exc.delete()

reset_case_study()

We then find the land occupation elementary flow - we need this to construct our new model.

In [ ]:
occupation = bd.get_node(name="Occupation, permanent crop", database="ecoinvent-3.10.1-biosphere")

We can create a new workspace (Brightway database), to keep our modifications in one place, and allow for clean attribution and change tracking of this specific experiment.

In [ ]:
land_use = bd.Database("Malaysian palm land use")
land_use.register()

Let's load the official data tables (extracted from the PDF using [tabula](https://tabula.technology/)):

In [ ]:
area = pd.read_csv("data/planted_area.csv", thousands=',')
area.set_index("State", inplace=True)
harvest = pd.read_csv("data/production.csv", thousands=',')

# They have the same "State" labels, so link them together
harvest.set_index("State", inplace=True)

# Get ready of extra rows only in one CSV
df = harvest.join(area).dropna()

# Go from absolute to fraction of total production
df['production_fraction_2022'] = df['2022'] / df['2022'].sum()

# Convert to the same metric as ecoinvent; note that we use a single conversion factor from fruit bunch to oil here
df['land_use_m2_per_kg'] = (
    df['Total']   # Hectare
    * 10000       # Hectare to square meter
    / df['2022']  # Tonnes
    / 3.94        # Conversion from oil production to fruit bunch production - ecoinvent number
    / 1000        # Tonnes to kilogram
)
df

Make sure that our Malaysian state labels are the same across our data sources (we will be matching against GIS data later).

A key aspect of successfully implement the tabular data paradigm in Brightway One will be leading users to proactively manage data quality. For example, one could specify a best guess value before looking at the input data or model results, and reasonable minimum and maximum values. These bounds can then be used both for automated data quality assessment and as checks preventing the import of new data outside the analysts expectations.

In [ ]:
gis = geopandas.read_file(bwr.geocollections['regions']['filepath'])
assert not set(df.index).difference(set(gis['name']))

## Building a new inventory graph

We will construct proxy datasets for palm fruit bunch production in each Malaysian state. For this limited case study, we will focus *only* on land use. However, in real analysis we could dive more into the production differences between small estates and industrialized plantations, and differences in transport distances, energy mixes, and other process efficiencies in the individual states.

We will construct a new market mix for land use, and link that to separate proxy processes for each state:

![image](img/graph.png)

In [ ]:
mix_process = land_use.new_node(
    name="Mix of land use across Malaysian states for 1 kg of palm fruit bunch production",
    year=2022,
    location=('regions', 'Kuala Lumpur'),
    type="process",
)
mix_process.save()
mix_product = land_use.new_node(
    name="Palm fruit production in Malaysia - 2022 market mix",
    year=2022,
    type="product",
    unit="kilogram",
)
mix_product.save()
mix_process.new_edge(
    amount=1,
    type="production",
    input=mix_product,
).save()

We now need to set the land occupation factor in the original ecoinvent dataset to zero - otherwise we would get double-counting.

In [ ]:
palm_ecoinvent = bd.get_node(
    database="ecoinvent-3.10.1-cutoff",
    name='palm fruit bunch production',
    location="MY"
)

In [ ]:
existing_land_use = next(exc for exc in palm_ecoinvent.biosphere() if exc.input == occupation)
existing_land_use['amount'] = 0
existing_land_use.save()
existing_land_use

And we add a link to our new proxy mix. Note that we set the *amount* here to 1 - we will calculate the land use our selves.

In [ ]:
palm_ecoinvent.new_edge(
    type="technosphere",
    amount=1,
    input=mix_product
).save()

## Creating the state-specific processes in a loop

We can simply iterate over the Pandas dataframe and create a new process and product for each Malaysian state, and add in:

* The fraction of total production from that state
* The relative land use of total production in that state

In [ ]:
for state, record in df.to_dict(orient="index").items():
    # Input data quality check
    if np.isnan(record['land_use_m2_per_kg']) or np.isnan(record['production_fraction_2022']):
        raise ValueError
    process = land_use.new_node(
        name=f"Land use proxy for palm fruit bunch production in {state}",
        year=2022,
        location=('regions', state),
        type="process",
    )
    process.save()
    product = land_use.new_node(
        name=f"Palm fruit production in {state}",
        year=2022,
        unit="kilogram",
        type="product",
    )
    product.save()
    process.new_edge(
        amount=1,
        type="production",
        input=product
    ).save()    
    process.new_edge(
        amount=record['land_use_m2_per_kg'],
        type="biosphere",
        input=occupation
    ).save()    
    mix_process.new_edge(
        type="technosphere",
        amount=record['production_fraction_2022'],
        input=product
    ).save()

## GIS Setup

Once we have defined our inventory, we need to do a bit more GIS setup to enable regionalized calculations.

In [ ]:
product = bd.get_node(name="Palm fruit production in Malaysia - 2022 market mix")

In [ ]:
bd.Database("Malaysian palm land use").set_geocollections()

In [ ]:
bwr.calculate_needed_intersections({product: 1}, ("GLAM", "land use"))

In [ ]:
bwr.intersections

In [ ]:
bwr.raster_as_extension_table('regions - ecoregions', 'palm', engine='rasterstats', overwrite=True)

In [ ]:
bwr.calculate_intersection('regions', 'regions - ecoregions', engine='geopandas')
bwr.calculate_intersection('regions - ecoregions', 'ecoregions', engine='geopandas')